# Part 5: すべてのソースを 1 つに

全社会議の前に、COO から「会社の状況を手早くまとめてほしい」と頼まれました。Fabric からの在庫リスク、在庫状況に関するアクティブなメールコミュニケーションのサマリ、会社ポリシーから読み解いたエスカレーションプロトコル、そして従業員向けの health benefits のハイライトを、すべて 1 つの回答にまとめる必要があります。

**🎯 ミッション**
- HR docs、health docs、Fabric IQ、Work IQ を組み合わせた **4 ソースの knowledge base** を構築する
- 構造化された製品データ、個人の職場コミュニケーション、ポリシードキュメント、health benefits を同時にまたがる単一のクエリを発行する

## Step 1: 環境変数をセットアップする

Azure AI Search、Azure OpenAI、Fabric リソースの設定を読み込みます。プロジェクトフォルダーの `.env` ファイルにこのパートで必要な値が入っています。

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
FABRIC_WORKSPACE_ID = os.environ["FABRIC_WORKSPACE_ID"]
FABRIC_ONTOLOGY_ID = os.environ["FABRIC_ONTOLOGY_ID"]

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

## Step 2: サインインと delegated token の取得

knowledge base の retrieval で Work IQ および Fabric IQ の knowledge source にあなたの ID を使うには、Microsoft 365 データと Fabric workspace の両方にアクセスできるサインイン済みユーザーの OAuth トークンが必要です。

下のセルを実行し、手順サイドバーに記載された資格情報で Azure にサインインしてください。成功すると Azure AI Search スコープの委任トークンを取得し、retrieval 時に `query_source_authorization` として使用するために保存します。

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(tenant_id=AZURE_TENANT_ID)
user_token = user_credential.get_token("https://search.azure.com/.default").token

print("Acquired token for logged-in user")

## Step 3: すべての knowledge source を作成する

この knowledge base 用に、これまでと同じ 2 つの search-index ソースに加え、Work IQ と Fabric Ontology のソースを追加して計 4 つを作成します。

* `hrdocs-knowledge-source`: `hrdocs` 検索インデックスを参照
* `healthdocs-knowledge-source`: `healthdocs` 検索インデックスを参照
* `workiq-knowledge-source`: 職場コンテキストのために Work IQ を参照
* `fabric-ontology-knowledge-source`: 構造化された業務データに使用される Fabric workspace と ontology を参照

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    FabricOntologyKnowledgeSource,
    FabricOntologyKnowledgeSourceParameters,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
FABRIC_KNOWLEDGE_SOURCE_NAME = "fabric-ontology-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    HR_KNOWLEDGE_SOURCE_NAME,
    HEALTH_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
    FABRIC_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="uid"),
                SearchIndexFieldReference(name="snippet_parent_id"),
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Zava Work IQ knowledge source",
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)

fabric_knowledge_source = FabricOntologyKnowledgeSource(
    name=FABRIC_KNOWLEDGE_SOURCE_NAME,
    description="Zava Fabric Ontology knowledge source",
    fabric_ontology_parameters=FabricOntologyKnowledgeSourceParameters(
        workspace_id=FABRIC_WORKSPACE_ID,
        ontology_id=FABRIC_ONTOLOGY_ID,
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=fabric_knowledge_source)
print("Knowledge sources created")

## Step 4: 統合 knowledge base を作成する

knowledge base は knowledge source 群を LLM と、retrieval の振る舞いに関する指示と組み合わせます。

このノートブックでは `outputMode` に `answerSynthesis` を指定し、アタッチしたモデルが質問に直接答えられるようにします。`retrievalReasoningEffort` は `low` に設定し、クエリプランニングと source 選択にモデルを利用します。

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-work-fabric-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Zava KB combining search indexes for HR and health documents, Work IQ for workplace context, and Fabric Ontology for products data",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for workplace context, Fabric Ontology for structured operational data, and search indexes for HR and health policy documents.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

## Step 5: knowledge base にクエリを投げる

4 つすべてのソースを同時にまたがる、全社会議向けの 4 つの観点を含む質問を投げます。

* _"Which product categories have the most critically low stock levels?"_ → **Fabric IQ**
* _"Do I have recent emails about the claw hammer stock situation?"_ → **Work IQ**
* _"Which company roles are in charge of budget management?"_ → `hrdocs`
* _"What mental health coverage does Zava offer employees?"_ → `healthdocs`

⏱️ Work IQ と Fabric IQ knowledge source の latency が大きいため、このサンプル質問への回答には約 1〜2 分かかります。

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FabricOntologyKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential
)

question = ("For the Friday all-hands, I need four things: "
            "1) Which Zava product categories have the most critically low stock levels? Check in Fabric product data. "
            "2) Do I have any recent work emails about the Professional Claw Hammer stock? "
            "3) Which Zava company roles are in charge of budget management? "
            "4) What mental health coverage does Zava offer employees? I want to mention our benefits briefly at the all-hands.")

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
        FabricOntologyKnowledgeSourceParams(
            knowledge_source_name=FABRIC_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=300,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_source_authorization=user_token,
)

# Extract synthesized answer from response messages
display(Markdown(result.response[0].content[0].text))


### activity log を確認する

今回の activity log では、`fabricOntology`、`workIQ`、`searchIndex` への呼び出しが、各ステップのクエリとともに表示されるはずです。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

knowledge base の多数のソースから取得されたさまざまな種類の reference が混在して表示されるはずです。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

上に表示された references を見て、4 つすべてのソース種別を特定してみましょう。

1. どの reference が **Fabric IQ**（構造化された製品エンティティ）から取得されましたか？
2. どの reference が **Work IQ**（あなたの M365 のメールや Teams）から取得されましたか？
3. どの reference が **HR インデックス**（ポリシードキュメント）から取得されましたか？
4. どの reference が **healthdocs**（health benefits ドキュメント）から取得されましたか？

## ✅ ミッション達成

**ラボ全体で構築したもの:**

* ✓ **4 ソースの Knowledge Base**: HR docs、health docs、構造化された Fabric の製品データ、ライブの M365 職場コミュニケーションにまたがる単一の KB。
* ✓ **並行マルチソース取得**: 1 つの質問をサブクエリに分解し、4 つすべてのソース種別へ同時にルーティング。
* ✓ **完結した agentic RAG パイプライン**: ビルド済みインデックスから Fabric ontology、個人メールまで、KB は 1 つの応答ですべてのソースを引用しつつ合成しました。